# Lab 03 – Reading and Writing Data with Python

| | |
|---|---|
| **Name** | Usman Hassan |
| **Roll number** | 1421-324032 |
| **Date** | 2 November 2026 |
| **AI tools used** | Claude (Anthropic) – helped write and check the code; I can explain every line. |

In [2]:
# data files were uploaded to Colab
import os
for name in ["students_marks.csv", "students_info.json", "university.db"]:
    print(name, "found" if os.path.exists(name) else "MISSING")

students_marks.csv found
students_info.json found
university.db found


## Task 1 – CSV

In [3]:
import csv

# 1. loading the marks file: every row becomes a dictionary, every value is text
with open("students_marks.csv", encoding="utf-8") as f:
    students = list(csv.DictReader(f))

subjects = ["Math", "Stats", "Python", "Databases", "English"]

# 2. converting the five marks to numbers and add the Average key
for s in students:
    for sub in subjects:
        s[sub] = int(s[sub])
    s["Average"] = round(sum(s[sub] for sub in subjects) / len(subjects), 2)

print(len(students), "students loaded")
print(students[0])

25 students loaded
{'RollNo': 'BSDS-01', 'Name': 'Ali Khan', 'Section': 'B', 'Math': 66, 'Stats': 80, 'Python': 81, 'Databases': 74, 'English': 69, 'Average': 74.0}


In [4]:
# 3. counting the students in each section with a dictionary
section_count = {}
for s in students:
    section_count[s["Section"]] = section_count.get(s["Section"], 0) + 1
print("Students per section:", dict(sorted(section_count.items())))

Students per section: {'A': 10, 'B': 12, 'C': 3}


In [5]:
# 4. students with 80 or more in Python -> top_python.csv (Name, Section, Python)
top_python = [s for s in students if s["Python"] >= 80]

with open("top_python.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["Name", "Section", "Python"],
                            extrasaction="ignore")
    writer.writeheader()
    writer.writerows(top_python)

print(len(top_python), "students have 80 or more in Python\n")
print(open("top_python.csv", encoding="utf-8").read())

11 students have 80 or more in Python

Name,Section,Python
Ali Khan,B,81
Sara Ahmed,A,82
Fatima Noor,A,88
Danish Iqbal,A,86
Maryam Shah,B,81
Talha Mehmood,A,82
Noor Fatima,B,80
Saad Anwar,A,89
Laiba Khan,C,87
Rabia Aslam,B,100
Imran Haider,B,87



## Task 2 – JSON

In [6]:
import json

with open("students_info.json", encoding="utf-8") as f:
    info = json.load(f)          # list of dictionaries

# 1. count students per city (City is nested inside Contact)
city_count = {}
for s in info:
    c = s["Contact"]["City"]
    city_count[c] = city_count.get(c, 0) + 1
print("Students per city:", dict(sorted(city_count.items())))

Students per city: {'Islamabad': 9, 'Lahore': 4, 'Murree': 2, 'Peshawar': 3, 'Rawalpindi': 7}


In [7]:
# 2. students who do NOT live in the hostel -> day_scholars.json (Name, City)
day_scholars = [{"Name": s["Name"], "City": s["Contact"]["City"]}
                for s in info if not s["Hostel"]]

with open("day_scholars.json", "w", encoding="utf-8") as f:
    json.dump(day_scholars, f, indent=2)

print(len(day_scholars), "day scholars saved to day_scholars.json")

15 day scholars saved to day_scholars.json


## Task 3 – Dates

In [8]:
from datetime import datetime, date

# 1. turn every Enrolled text (dd/mm/yyyy) into a real date
for s in info:
    s["EnrolledDate"] = datetime.strptime(s["Enrolled"], "%d/%m/%Y").date()

print(info[0]["Enrolled"], "->", info[0]["EnrolledDate"].isoformat())

02/09/2024 -> 2024-09-02


In [9]:
# 2. how many students joined in each intake (year-month)
intake_count = {}
for s in info:
    key = s["EnrolledDate"].strftime("%Y-%m")
    intake_count[key] = intake_count.get(key, 0) + 1
for intake, n in sorted(intake_count.items()):
    print(intake, ":", n, "students")

2024-09 : 21 students
2025-02 : 4 students


In [10]:
# 3. days the first student has been enrolled on 1 November 2026
first = info[0]
days = (date(2026, 11, 1) - first["EnrolledDate"]).days
print(first["Name"], "has been enrolled for", days, "days on 1 November 2026")

Ali Khan has been enrolled for 790 days on 1 November 2026


In [11]:
# 4. save Name + Enrolled date with a custom encoder so dates become ISO text
class DateEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (date, datetime)):
            return obj.isoformat()
        return super().default(obj)

enrolled = [{"Name": s["Name"], "Enrolled": s["EnrolledDate"]} for s in info]
with open("enrolled.json", "w", encoding="utf-8") as f:
    json.dump(enrolled, f, indent=2, cls=DateEncoder)

print(len(enrolled), "records saved to enrolled.json")
print(json.dumps(enrolled[:2], indent=2, cls=DateEncoder))

25 records saved to enrolled.json
[
  {
    "Name": "Ali Khan",
    "Enrolled": "2024-09-02"
  },
  {
    "Name": "Sara Ahmed",
    "Enrolled": "2024-09-02"
  }
]


## Task 4 – SQL with SQLAlchemy

In [12]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///lab3.db")
Base = declarative_base()

# 1. one class = one table: students (id, Name, Section, Python)
class Student(Base):
    __tablename__ = "students"
    id = Column(Integer, primary_key=True)
    Name = Column(String)
    Section = Column(String)
    Python = Column(Integer)

    def __repr__(self):
        return f"<{self.Name}, {self.Section}, {self.Python}>"

Base.metadata.drop_all(engine)       # start fresh every time the notebook runs
Base.metadata.create_all(engine)
session = sessionmaker(bind=engine)()

session.add_all([Student(Name=s["Name"], Section=s["Section"], Python=s["Python"])
                 for s in students])
session.commit()

# 2. number of rows
print(session.query(Student).count(), "rows in lab3.db")

25 rows in lab3.db


In [13]:
# 3. Section B students with Python 70 or more, highest first
section_b = (session.query(Student)
             .filter(Student.Section == "B", Student.Python >= 70)
             .order_by(Student.Python.desc())
             .all())
print(len(section_b), "Section B students with Python >= 70:")
for st in section_b:
    print(f"  {st.Name:<16} {st.Python}")

6 Section B students with Python >= 70:
  Rabia Aslam      100
  Imran Haider     87
  Ali Khan         81
  Maryam Shah      81
  Noor Fatima      80
  Iqra Javed       70


## Task 5 – The fees database

In [14]:
from sqlalchemy import text

uni = create_engine("sqlite:///university.db")
names = {s["RollNo"]: s["Name"] for s in students}     # RollNo -> Name

with uni.connect() as conn:
    late = conn.execute(text(
        "SELECT RollNo, DueDate, PaidOn FROM fees "
        "WHERE PaidOn IS NOT NULL AND PaidOn > DueDate")).fetchall()
    unpaid = conn.execute(text(
        "SELECT RollNo FROM fees WHERE PaidOn IS NULL")).fetchall()
    collected = conn.execute(text(
        "SELECT SUM(Amount) FROM fees WHERE PaidOn IS NOT NULL")).scalar()

print("Paid late:")
for roll, due, paid_on in late:
    days_late = (date.fromisoformat(paid_on) - date.fromisoformat(due)).days
    print(f"  {names[roll]:<16} {days_late} days late")

print("\nNot paid yet:", ", ".join(names[r[0]] for r in unpaid))
print(f"\nTotal collected: Rs {collected:,}")

Paid late:
  Bilal Hussain    10 days late
  Usman Tariq      12 days late
  Asad Mirza       6 days late
  Hassan Butt      15 days late
  Sana Rauf        13 days late

Not paid yet: Hamza Raza, Faisal Qureshi

Total collected: Rs 2,585,000


## Task 6 – One packet and Power BI

In [15]:
# 1. join the three sources on RollNo
info_by_roll = {s["RollNo"]: s for s in info}

with uni.connect() as conn:
    fees = {r[0]: (r[1], r[2]) for r in conn.execute(
        text("SELECT RollNo, DueDate, PaidOn FROM fees")).fetchall()}

def fee_status(due, paid_on):
    if paid_on is None:
        return "Unpaid"
    return "Late" if paid_on > due else "On time"     # ISO text compares like dates

packet = []
for s in students:
    roll = s["RollNo"]
    packet.append({
        "RollNo": roll,
        "Name": s["Name"],
        "Section": s["Section"],
        "City": info_by_roll[roll]["Contact"]["City"],
        "Hostel": info_by_roll[roll]["Hostel"],
        "Average": s["Average"],
        "FeeStatus": fee_status(*fees[roll]),
    })

with open("students_full.json", "w", encoding="utf-8") as f:
    json.dump(packet, f, indent=2)

print(len(packet), "students saved to students_full.json")
print(packet[0])

25 students saved to students_full.json
{'RollNo': 'BSDS-01', 'Name': 'Ali Khan', 'Section': 'B', 'City': 'Islamabad', 'Hostel': False, 'Average': 74.0, 'FeeStatus': 'On time'}


In [16]:
# check numbers for the Power BI answers below
city_avg = {}
for p in packet:
    city_avg.setdefault(p["City"], []).append(p["Average"])
for c, vals in sorted(city_avg.items(), key=lambda kv: -sum(kv[1]) / len(kv[1])):
    print(f"{c:<11} average {sum(vals)/len(vals):.2f}  ({len(vals)} students)")

status_count = {}
for p in packet:
    status_count[p["FeeStatus"]] = status_count.get(p["FeeStatus"], 0) + 1
print("\nFee status:", status_count)

Lahore      average 76.90  (4 students)
Islamabad   average 73.80  (9 students)
Murree      average 71.10  (2 students)
Rawalpindi  average 66.46  (7 students)
Peshawar    average 59.00  (3 students)

Fee status: {'On time': 18, 'Late': 5, 'Unpaid': 2}


### Task 6 – Power BI and answers

**Power BI report** (lab03_report.pbix / lab03_report.pdf): `students_full.json` loaded through Get data > JSON > To Table > expand; a clustered column chart of **Average by City** (aggregation set to *Average*), a **Card** showing the number of students (25), and a **FeeStatus** slicer.

**Answers**
- **Lahore** has the highest average marks: **76.90** (4 students). Next are Islamabad (73.80), Murree (71.10), Rawalpindi (66.46) and Peshawar (59.00).
- **2 students** have not paid their fee: Hamza Raza and Faisal Qureshi. Another 5 paid late and 18 paid on time.